# BR07 — Freeze a single-clinician ECG report review

Run after BR05A section A using the same completed BR04B child config. CPU only; no generators are loaded. Read `clinical_evaluation_plan.md` first. One ECG-qualified clinician first interprets query ECGs without generated reports. BR08 releases reports only after these interpretations are complete. Clinical disagreements later receive a separate same-clinician adjudication pass.

The default is a 10-ECG **calibration pilot**, two models and the two primary arms (40 reports). Do not treat this as the main powered clinical study. Main samples need a documented sample-size justification. No clinical judgment is generated automatically. Existing directories are preserved; after interruption reopen the same directory rather than preparing it again.

This version copies verified historical PNGs from the completed BR05A modern reconstruction. It does not rerender ECGs, so the CPU review kernel does not need the original Matplotlib version. The selected modern generators must share the same historical image hashes. Upload `clinical_review/archived_inputs.py` alongside this notebook; keep the original `workflow.py` and `extensions/rendering_audit.py` unchanged.


> **Clinical review:** original helpers are included in `clinical_review/`. Completed reviewer data are not distributed. See `docs/RUN_ORDER.md` and `docs/REPRODUCIBILITY.md`.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None:
    raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
# Use the EXACT completed BR04B child config. Change this for later cohorts.
CONFIG_PATH = Path('/path/to/repository/outputs/real_run_001/'
                   'experiment02_v1/context_fit/v1/internal_val/config.json')
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Set CONFIG_PATH to the actual BR04B context-fit child config: ' + str(CONFIG_PATH))
CFG = load_config(CONFIG_PATH)
parent_setup(CFG)
RUN = Path(CFG['run_dir'])
REVIEW_VERSION = 'calibration_v1'  # Same value in BR07, BR08 and BR09. New version for main review.
REVIEW_DIR = RUN / 'clinical_review' / CFG['cohort'] / REVIEW_VERSION
from clinical_review.workflow import progress
print('Cohort:', CFG['cohort'], '| review:', REVIEW_DIR)
if CFG['cohort'] == 'internal_val':
    print('Validation review is calibration/exploratory, not held-out clinical validation.')


In [ ]:
from generation import folder_for
from clinical_review.archived_inputs import prepare_from_reconstruction
SELECTED_MODELS = ('llama_ecg', 'qwen38')  # Matched modern rendering; choose before human rating.
TOKEN_BUDGETS = {'pulse': 384, 'llama_ecg': 384, 'llama_base': 384, 'qwen38': 1024}
FOLDERS = {m: folder_for(CFG, m, False, TOKEN_BUDGETS[m]) for m in SELECTED_MODELS}
ARMS = ('bidirectional_pool', 'bidirectional_graph')
N_QUERIES = 10
STAGE = 'calibration'  # 'main' only after rubric calibration and sample-size justification.
REVIEWER_ID = 'ECG_reviewer_01'
REVIEWER_EXPERTISE = ''  # REQUIRED: specialty, ECG-reading experience, development involvement.
SAMPLE_SIZE_JUSTIFICATION = 'Ten validation ECGs for rubric and workflow calibration only.'
SOURCE_ACCESS_POLICY = 'Exact generator ECG image only; absolute voltage/calibration limitations recorded.'
ALLOW_PARTIAL = False  # If True, review successful matched cases; all exclusions remain in coverage.
HISTORICAL_RENDER_DIR = (RUN / 'audits' / 'historical_rendering' / CFG['cohort'] / 'v1' / 'modern')
print('Verified historical image archive:', HISTORICAL_RENDER_DIR)
# Do not substitute RUN/protocol.json or the shared queries/.../images directory.
print('Frozen source folders:', {m: str(p) for m, p in FOLDERS.items()})
if REVIEW_DIR.exists():
    existing = read(REVIEW_DIR / 'PRIVATE/protocol.json')
    if set(existing['models']) != set(FOLDERS) or existing['reviewer_id'] != REVIEWER_ID:
        raise ValueError('Existing package uses different models/reviewer. Preserve it and select a new REVIEW_VERSION in BR07–BR09.')
    print('Existing package preserved. Resume below or in BR08:', progress(REVIEW_DIR))
else:
    print(prepare_from_reconstruction(CFG, FOLDERS, REVIEW_DIR, HISTORICAL_RENDER_DIR,
        n_queries=N_QUERIES, arms=ARMS,
        seed=CFG['seed'], reviewer_id=REVIEWER_ID, reviewer_expertise=REVIEWER_EXPERTISE,
        stage=STAGE, sample_size_justification=SAMPLE_SIZE_JUSTIFICATION,
        allow_partial=ALLOW_PARTIAL, source_access_policy=SOURCE_ACCESS_POLICY))


## Clinician: interpret each ECG before opening reports

Only PUBLIC is reviewer-facing. Do not inspect PRIVATE, model outputs, automatic labels or analysis. Repeat the following two cells per case. Record a genuine independent interpretation; if uninterpretable, describe why. Saved drafts survive kernel restarts. Once BR08 submits interpretations, they cannot be rewritten after seeing reports.

In [ ]:
from IPython.display import display, Image
from clinical_review.workflow import save_reference
cases = read(REVIEW_DIR / 'PUBLIC/reference_cases.json')
remaining = [r for r in cases if not (REVIEW_DIR / 'PUBLIC/reference_drafts' / (r['case_id'] + '.json')).exists()]
CASE = remaining[0] if remaining else None
print(progress(REVIEW_DIR))
if CASE is None:
    print('All independent interpretations saved. Continue to BR08.')
else:
    print('Current anonymous case:', CASE['case_id'])
    display(Image(filename=str(REVIEW_DIR / 'PUBLIC' / CASE['image'])))


In [ ]:
# HUMAN ENTRY ONLY. Edit these values for the displayed ECG, then set SAVE_REFERENCE=True.
SAVE_REFERENCE = False
INTERPRETABLE = None  # True or False, never inferred automatically.
INDEPENDENT_ECG_INTERPRETATION = None  # Clinician's own assessment, or reason interpretation is impossible.
LIMITATIONS = ''
if SAVE_REFERENCE:
    if CASE is None: raise ValueError('No current case')
    save_reference(REVIEW_DIR, CASE['case_id'], reviewer_id=REVIEWER_ID,
        interpretable=INTERPRETABLE, independent_ecg_interpretation=INDEPENDENT_ECG_INTERPRETATION,
        limitations=LIMITATIONS)
    print('Saved:', CASE['case_id'], '| rerun the display cell for the next case')
else:
    print('No clinical judgment saved. Enter the clinician assessment before enabling save.')


When all cases are saved, run BR08 with exactly the same CONFIG_PATH and REVIEW_VERSION. For a main review use a fresh version (for example `main_v1`) with the chosen cohort, sample-size rationale and reviewer expertise fixed before preparation. Do not pool calibration and main ratings.